In [18]:
%%writefile test_engine.py

import pytest
import torch

class Value:
    def __init__(self, val, prev=(), op='', label=''):
        self.val   = val
        self.prev  = set(prev)
        self.op    = op
        self.label = label
        self.grad  = 0.0
        self._backward = lambda: None

    def __add__(self, other):
        other  = other if isinstance(other, Value) else Value(other)
        output = Value(self.val + other.val, (self, other), op="+")
        def backward():
            self.grad  += 1.0 * output.grad
            other.grad += 1.0 * output.grad
        output._backward = backward
        return output

    def __mul__(self, other):
        other  = other if isinstance(other, Value) else Value(other)
        output = Value(self.val * other.val, (self, other), op="*")
        def backward():
            self.grad  += other.val * output.grad
            other.grad += self.val  * output.grad
        output._backward = backward
        return output

    def __pow__(self, other):
        assert isinstance(other, (int, float))
        output = Value(self.val ** other, (self,), op=f"**{other}")
        def backward():
            self.grad += other * (self.val ** (other - 1)) * output.grad
        output._backward = backward
        return output

    def relu(self):
        output = Value(0 if self.val < 0 else self.val, (self,), op="Relu")
        def backward():
            self.grad += (output.val > 0) * output.grad
        output._backward = backward
        return output

    def __neg__(self):          return self * -1
    def __radd__(self, other):  return self + other
    def __sub__(self, other):   return self + (-other)
    def __rsub__(self, other):  return other + (-self)
    def __truediv__(self, other):  return self * (other ** -1)
    def __rtruediv__(self, other): return other * (self ** -1)

    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v.prev:
                    build(child)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

TOL = 1e-5

def pt(x, requires_grad=True):
    return torch.tensor(float(x), requires_grad=requires_grad)

class TestForwardPass:
    def test_addition(self):
        assert (Value(3.0) + Value(4.0)).val == pytest.approx(7.0, abs=TOL)
    def test_multiplication(self):
        assert (Value(3.0) * Value(4.0)).val == pytest.approx(12.0, abs=TOL)
    def test_subtraction(self):
        assert (Value(7.0) - Value(3.0)).val == pytest.approx(4.0, abs=TOL)
    def test_division(self):
        assert (Value(10.0) / Value(4.0)).val == pytest.approx(2.5, abs=TOL)
    def test_power(self):
        assert (Value(3.0) ** 3).val == pytest.approx(27.0, abs=TOL)
    def test_relu_positive(self):
        assert Value(5.0).relu().val == pytest.approx(5.0, abs=TOL)
    def test_relu_negative(self):
        assert Value(-3.0).relu().val == pytest.approx(0.0, abs=TOL)

class TestBackwardVsPytorch:
    def _check(self, our, pt_t):
        assert our.grad == pytest.approx(pt_t.grad.item(), abs=TOL)

    def test_grad_add(self):
        a, b = Value(2.0), Value(5.0)
        (a + b).backward()
        ta, tb = pt(2.0), pt(5.0)
        (ta + tb).backward()
        self._check(a, ta); self._check(b, tb)

    def test_grad_mul(self):
        a, b = Value(3.0), Value(4.0)
        (a * b).backward()
        ta, tb = pt(3.0), pt(4.0)
        (ta * tb).backward()
        self._check(a, ta); self._check(b, tb)

    def test_grad_pow(self):
        a = Value(3.0)
        (a ** 4).backward()
        ta = pt(3.0)
        (ta ** 4).backward()
        self._check(a, ta)

    def test_grad_relu_positive(self):
        a = Value(3.0)
        a.relu().backward()
        ta = pt(3.0)
        torch.relu(ta).backward()
        self._check(a, ta)

    def test_grad_relu_negative(self):
        a = Value(-2.0)
        a.relu().backward()
        ta = pt(-2.0)
        torch.relu(ta).backward()
        self._check(a, ta)

    def test_grad_mse_loss(self):
        w1, w2, b = Value(0.5), Value(-1.0), Value(2.0)
        y = Value(2.0) * w1 + Value(3.0) * w2 + b
        ((y - Value(8.0)) ** 2).backward()
        tw1, tw2, tb = pt(0.5), pt(-1.0), pt(2.0)
        ty = pt(2.0, False) * tw1 + pt(3.0, False) * tw2 + tb
        ((ty - 8.0) ** 2).backward()
        self._check(w1, tw1); self._check(w2, tw2); self._check(b, tb)

class TestGradientDescent:
    def test_loss_decreases(self):
        w1, w2, b = Value(0.5), Value(-1.0), Value(2.0)
        prev_loss = float('inf')
        for _ in range(20):
            w1.grad = w2.grad = b.grad = 0.0
            y = Value(2.0) * w1 + Value(3.0) * w2 + b
            loss = (y - Value(8.0)) ** 2
            loss.backward()
            w1.val -= 0.01 * w1.grad
            w2.val -= 0.01 * w2.grad
            b.val  -= 0.01 * b.grad
            assert loss.val < prev_loss
            prev_loss = loss.val

    def test_loss_converges(self):
        w1, w2, b = Value(0.5), Value(-1.0), Value(2.0)
        for _ in range(500):
            w1.grad = w2.grad = b.grad = 0.0
            y = Value(2.0) * w1 + Value(3.0) * w2 + b
            loss = (y - Value(8.0)) ** 2
            loss.backward()
            w1.val -= 0.05 * w1.grad
            w2.val -= 0.05 * w2.grad
            b.val  -= 0.05 * b.grad
        assert loss.val < 0.01

Overwriting test_engine.py


In [ ]:
!pytest test_engine.py -v

============================= test session starts =============================
platform win32 -- Python 3.12.3, pytest-9.0.3, pluggy-1.6.0 -- C:\Users\sriva\anaconda3\envs\vatsavx\python.exe
cachedir: .pytest_cache
rootdir: c:\praveensir\basic_code_base
plugins: anyio-4.6.2
collecting ... collected 15 items

test_engine.py::TestForwardPass::test_addition PASSED                    [  6%]
test_engine.py::TestForwardPass::test_multiplication PASSED              [ 13%]
test_engine.py::TestForwardPass::test_subtraction PASSED                 [ 20%]
test_engine.py::TestForwardPass::test_division PASSED                    [ 26%]
test_engine.py::TestForwardPass::test_power PASSED                       [ 33%]
test_engine.py::TestForwardPass::test_relu_positive PASSED               [ 40%]
test_engine.py::TestForwardPass::test_relu_negative PASSED               [ 46%]
test_engine.py::TestBackwardVsPytorch::test_grad_add PASSED              [ 53%]
test_engine.py::TestBackwardVsPytorch::test_grad_

: 